```
CADERNO DE ANÁLISE DE DADOS & INTELIGÊNCIA COMERCIAL (ETL + DUCKDB + SQL)
Autora: Caroline Rocha
Objetivo:
Tratar dados brutos de faturamento (7.873 transações),
anonimizar identificadores,
calcular KPIs comerciais,
Curva ABC (Pareto) e
métricas de recorrência com DuckDB.
```

In [16]:
import duckdb
import pandas as pd
import numpy as np

In [17]:
# 1. Leitura direta do CSV anonimizado
df = pd.read_csv("faturamento.csv")

In [18]:
df.head(5)

,AnoMes,ANO,MES,DataEmissao,NumeroRPS,NumeroNF,Categoria,Cliente,CNPJ,Vendedor,ValorTotalServico,ImpostosRetidos,ValorLiquido
0,202409,2024,setembro,2024-02-09 00:00:00,7922,15168,SERVICOS PRESTADOS PREVIDENCIARIO,Northfield Harriet Edwards,3923010698,Leonard Bennett,394.50,0,394.50
1,202409,2024,setembro,2024-02-09 00:00:00,7923,15169,SERVICOS PRESTADOS PREVIDENCIARIO,Northfield Nathaniel Kensington,3661163394,Leonard Bennett,451.71,0,451.71
2,202409,2024,setembro,2024-03-09 00:00:00,7924,15170,SERVICOS PRESTADOS PREVIDENCIARIO,Northfield Katherine Iverson,4524189105,Adrian Anderson,2529.78,0,2529.78
3,202409,2024,setembro,2024-03-09 00:00:00,7925,15171,SERVICOS PRESTADOS PREVIDENCIARIO,Northfield Eleanor Johnson,2975183195,Leonard Bennett,283.16,0,283.16
4,202409,2024,setembro,2024-03-09 00:00:00,7926,15172,SERVICOS PRESTADOS PREVIDENCIARIO,Northfield Alexandre Dawson,3537596581,Adrian Anderson,567.25,0,567.25


In [19]:
# Converte o DataFrame para Parquet
df.to_parquet("faturamento.parquet", index=False)
print("Arquivo faturamento.parquet gerado com sucesso!")

Arquivo faturamento.parquet gerado com sucesso!


In [20]:
# Conecta ao DuckDB em memória
con = duckdb.connect()


In [21]:
# --------------------------------------------------------------------------------------
# ETAPA 1: CARREGAMENTO E ENTENDIMENTO DA BASE DE DADOS
# --------------------------------------------------------------------------------------
# A base foi tratada a partir de um CSV' do ERP original.
# O formato Parquet foi escolhido por ser colunar, compacto e muito mais veloz para consultas SQL.

caminho_dados = "faturamento.parquet"
con = duckdb.connect()

In [22]:
# Registrando o arquivo Parquet diretamente como tabela SQL no DuckDB sem carregar tudo na memória
con.execute(f"CREATE VIEW IF NOT EXISTS faturamento AS SELECT * FROM read_parquet('{caminho_dados}')")

print("Base carregada com sucesso!")
print("Estrutura das colunas:")
print(con.execute("DESCRIBE faturamento").df())

Base carregada com sucesso!
Estrutura das colunas:
          column_name column_type null   key default extra
0              AnoMes      BIGINT  YES  None    None  None
1                 ANO      BIGINT  YES  None    None  None
2                 MES     VARCHAR  YES  None    None  None
3         DataEmissao     VARCHAR  YES  None    None  None
4           NumeroRPS      BIGINT  YES  None    None  None
5            NumeroNF      BIGINT  YES  None    None  None
6           Categoria     VARCHAR  YES  None    None  None
7             Cliente     VARCHAR  YES  None    None  None
8                CNPJ      BIGINT  YES  None    None  None
9            Vendedor     VARCHAR  YES  None    None  None
10  ValorTotalServico      DOUBLE  YES  None    None  None
11    ImpostosRetidos      BIGINT  YES  None    None  None
12       ValorLiquido      DOUBLE  YES  None    None  None


In [23]:
# --------------------------------------------------------------------------------------
# ETAPA 2: KPIS GERAIS DO NEGÓCIO (BIG NUMBERS)
# --------------------------------------------------------------------------------------
# Aqui consolidamos a visão macro da empresa entre 2021 e 2024.
# Métricas calculadas:
# - Volume total de faturas emitidas
# - Faturamento bruto (ValorTotalServico) e líquido (ValorLiquido)
# - Total de impostos retidos na fonte
# - Ticket médio por fatura
# - Quantidade de clientes únicos atendidos e vendedores ativos

query_kpis = """
SELECT
    COUNT(*) AS total_faturas,
    ROUND(SUM(ValorTotalServico), 2) AS faturamento_bruto,
    ROUND(SUM(ImpostosRetidos), 2) AS impostos_retidos,
    ROUND(SUM(ValorLiquido), 2) AS faturamento_liquido,
    ROUND(AVG(ValorTotalServico), 2) AS ticket_medio,
    COUNT(DISTINCT Cliente) AS total_clientes,
    COUNT(DISTINCT Vendedor) AS total_franqueados
FROM faturamento
"""

In [24]:
df_kpis = con.execute(query_kpis).df()
print("\n--- KPIs GERAIS DO NEGÓCIO ---")
print(df_kpis)


--- KPIs GERAIS DO NEGÓCIO ---
   total_faturas  faturamento_bruto  impostos_retidos  faturamento_liquido  \
0           7873        38970649.61         1989808.0          36980813.05   

   ticket_medio  total_clientes  total_franqueados  
0       4949.91             487                 61  


In [25]:
# --------------------------------------------------------------------------------------
# ETAPA 3: CURVA ABC / PARETO (REGRA DOS 80/20)
# --------------------------------------------------------------------------------------
# Qual a concentração da receita nos principais clientes?
# Regra clássica de Pareto:
# - Classe A: clientes que somados representam até 80% do faturamento total.
# - Classe B: clientes que representam os 15% seguintes (80% a 95%).
# - Classe C: clientes da 'cauda longa', que representam os últimos 5% da receita.

query_pareto = """
WITH faturamento_por_cliente AS (
    SELECT
        Cliente,
        COUNT(*) AS qtd_compras,
        ROUND(SUM(ValorTotalServico), 2) AS total_faturado
    FROM faturamento
    GROUP BY Cliente
),
acumulado AS (
    SELECT
        Cliente,
        qtd_compras,
        total_faturado,
        SUM(total_faturado) OVER (ORDER BY total_faturado DESC) AS receita_acumulada,
        SUM(total_faturado) OVER () AS receita_total
    FROM faturamento_por_cliente
)
SELECT
    Cliente,
    qtd_compras,
    total_faturado,
    ROUND((receita_acumulada / receita_total) * 100, 2) AS pct_acumulado,
    CASE
        WHEN (receita_acumulada / receita_total) <= 0.80 THEN 'Classe A (80% da receita)'
        WHEN (receita_acumulada / receita_total) <= 0.95 THEN 'Classe B (15% da receita)'
        ELSE 'Classe C (5% cauda longa)'
    END AS classificacao_abc
FROM acumulado
ORDER BY total_faturado DESC
"""

In [26]:
df_pareto = con.execute(query_pareto).df()
print("\n--- TOP 10 CLIENTES (PARETO) ---")
print(df_pareto.head(10))


--- TOP 10 CLIENTES (PARETO) ---
                         Cliente  qtd_compras  total_faturado  pct_acumulado  \
0      Northfield Olivia Iverson           42      4103176.83          10.53   
1     Northfield Julian Anderson           38      2892576.38          17.95   
2     Northfield Adrian Harrison            3      1233300.88          21.12   
3      Northfield Hannah Iverson           34       720616.40          22.97   
4     Northfield Eleanor Edwards           53       664319.32          24.67   
5   Northfield Florence Anderson           17       647694.02          26.33   
6    Northfield Rebecca Anderson           21       578211.29          27.82   
7   Northfield Sebastian Bennett           40       504892.34          29.11   
8  Northfield Lillian Kensington           14       504872.27          30.41   
9      Northfield Yvonne Johnson           40       470698.89          31.61   

           classificacao_abc  
0  Classe A (80% da receita)  
1  Classe A (80% da rec

In [27]:
# Resumo de clientes por classe ABC
resumo_abc = con.execute(f"""
WITH p AS ({query_pareto})
SELECT
    classificacao_abc,
    COUNT(*) AS total_clientes,
    ROUND(SUM(total_faturado), 2) AS faturamento_classe,
    ROUND(SUM(total_faturado) * 100.0 / (SELECT SUM(total_faturado) FROM p), 2) AS pct_do_faturamento
FROM p
GROUP BY classificacao_abc
ORDER BY classificacao_abc
""").df()
print("\n--- DISTRIBUIÇÃO DAS CLASSES ABC ---")
print(resumo_abc)


--- DISTRIBUIÇÃO DAS CLASSES ABC ---
           classificacao_abc  total_clientes  faturamento_classe  \
0  Classe A (80% da receita)             112         31138203.97   
1  Classe B (15% da receita)             142          5871673.15   
2  Classe C (5% cauda longa)             233          1960772.49   

   pct_do_faturamento  
0               79.90  
1               15.07  
2                5.03  


In [28]:
# --------------------------------------------------------------------------------------
# ETAPA 4: ANÁLISE TEMPORAL (EVOLUÇÃO ANUAL E MENSAL)
# --------------------------------------------------------------------------------------
# Medir sazonalidade e taxa de crescimento anual (Year-over-Year).

query_temporal = """
SELECT
    ANO,
    COUNT(*) AS qtd_faturas,
    ROUND(SUM(ValorTotalServico), 2) AS faturamento_anual,
    ROUND(AVG(ValorTotalServico), 2) AS ticket_medio_anual,
    COUNT(DISTINCT Cliente) AS clientes_ativos
FROM faturamento
GROUP BY ANO
ORDER BY ANO
"""

In [29]:
df_temporal = con.execute(query_temporal).df()
print("\n--- EVOLUÇÃO ANUAL ---")
print(df_temporal)


--- EVOLUÇÃO ANUAL ---
    ANO  qtd_faturas  faturamento_anual  ticket_medio_anual  clientes_ativos
0  2021         1577         7380100.62             4679.84              251
1  2022         2354        12641916.77             5370.40              311
2  2023         2293        12895364.14             5623.80              287
3  2024         1649         6053268.08             3670.87              223


In [30]:
# --------------------------------------------------------------------------------------
# ETAPA 5: PERFORMANCE POR FRANQUEADO / VENDEDOR
# --------------------------------------------------------------------------------------
# Avaliar os maiores geradores de receita da rede e distribuição de volume.

query_franqueados = """
SELECT
    Vendedor AS Franqueado,
    COUNT(*) AS total_vendas,
    COUNT(DISTINCT Cliente) AS carteira_clientes,
    ROUND(SUM(ValorTotalServico), 2) AS receita_total,
    ROUND(AVG(ValorTotalServico), 2) AS ticket_medio
FROM faturamento
GROUP BY Vendedor
ORDER BY receita_total DESC
"""

In [31]:
df_franqueados = con.execute(query_franqueados).df()
print("\n--- TOP 5 FRANQUEADOS EM FATURAMENTO ---")
print(df_franqueados.head(5))

print("\nAnálise concluída com sucesso!")


--- TOP 5 FRANQUEADOS EM FATURAMENTO ---
           Franqueado  total_vendas  carteira_clientes  receita_total  \
0   Katherine Bennett           667                 68     6413681.00   
1     Adrian Anderson           752                 44     4098634.13   
2      Olivia Bennett           712                113     3736649.64   
3  Charlotte Anderson           381                 15     3499764.24   
4      Felix Anderson           232                 11     3425082.96   

   ticket_medio  
0       9615.71  
1       5450.31  
2       5248.10  
3       9185.73  
4      14763.29  

Análise concluída com sucesso!
